# Solar PPA microgrid

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bizarc/cfdl/blob/main/examples/notebooks/01_energy_solar_microgrid.ipynb)

A solar-plus-storage microgrid with a PPA revenue contract, degradation, O&M escalation, ITC/PTC tax attributes and MACRS depreciation, financed with sculpted debt.

This notebook uses one of the benchmark models, which CFDL validates against an independent reference to the penny.

In [ ]:
# On Colab, install the SDK and fetch the models this notebook reads.
# Inside a checkout both are already present and this cell does nothing.
import subprocess, sys
from pathlib import Path

REPO = "https://github.com/bizarc/cfdl"


def repo_root() -> Path:
    """The checkout holding benchmarks/ and packs/, cloning it if need be.

    Searching a bounded set of ancestors means a plain `python` run outside a
    checkout fails with an explanation rather than walking to the filesystem
    root. On a hosted runtime there is no checkout to find, so fetch one.
    """
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "Cargo.toml").exists() and (candidate / "packs").is_dir():
            return candidate

    if "google.colab" not in sys.modules:
        raise RuntimeError(
            f"No CFDL checkout found above {here}. This notebook reads a model "
            f"from benchmarks/ and pack definitions from packs/, so run it "
            f"inside a clone of {REPO}."
        )

    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "cfdl-sdk[viz]"], check=True)

    # Packs and benchmark models track the engine, so take the checkout at the
    # tag matching the wheel pip just resolved. `main` runs ahead of the last
    # release and its packs may use metric ops the released engine rejects.
    import importlib
    from importlib.metadata import PackageNotFoundError, version

    importlib.invalidate_caches()
    try:
        tag = f"v{version('cfdl-sdk')}"
    except PackageNotFoundError:
        tag = None

    clone = ["git", "clone", "--depth", "1", "-q", REPO]
    target = Path("/content/cfdl")
    if not target.exists():
        pinned = tag is not None and not subprocess.run(
            clone + ["--branch", tag, str(target)]
        ).returncode
        if not pinned:
            # A dev or pre-release wheel has no matching tag; main is the best
            # available, and the notebook may fail if the two have diverged.
            print(f"warning: no {tag} tag for this SDK build; falling back to main.")
            subprocess.run(clone + [str(target)], check=True)
    return target


ROOT = repo_root()
PACKS = ROOT / "packs"

import cfdl_sdk


## Compile

Compile the model directory to IR.

In [ ]:
model_dir = ROOT / "benchmarks/energy/solar_ppa_microgrid"
model = cfdl_sdk.compile(model_dir, packs_dir=PACKS)
print("streams:", len(model.ir["streams"]))

## Run

Run with the benchmark's configuration and apply the `energy` pack's domain metrics.

In [ ]:
results = model.run(
    config=str(model_dir / "run.json"),
    pack="energy",
)
print("status:", results.status, "| warnings:", len(results.warnings))

## Cash flows

The engine returns per-period signed cash flows; `cashflows()` gives a wide DataFrame indexed by period.

In [ ]:
cf = results.cashflows()
print('shape:', cf.shape)
cf.head()

In [ ]:
# Requires the [viz] extra (pip install cfdl-sdk[viz]).
results.plot.cumulative()

## Metrics

Core metrics (NPV/IRR/MOIC/...) plus the pack's domain metrics, with their source labeled.

In [ ]:
results.metrics_frame()

## What-if

Re-run at a higher discount rate and compare NPV.

In [ ]:
base = results.metrics()["model.npv"]
stressed = model.run(config={"deterministic": {"annual_discount_rate": 0.10}}, pack="energy")
print(f"NPV @ base: {base:,.0f}")
print(f"NPV @ 10%: {stressed.metrics()['model.npv']:,.0f}")

## Extended analysis — verify, decompose, cover

The same discipline an agent uses: don't trust a series, interrogate it.

In [ ]:
# Verify the degradation convention: annual PPA revenue should grow at a constant
# escalation-net-of-degradation rate. pandas makes the check one line.
ppa_year = cf["stream.energy.ppa.revenue"].groupby(cf.index.year).sum()
ppa_year.pct_change().dropna().round(4).unique()

A constant ~1.49% — exactly `(1 + 2% escalation) x (1 - 0.5% degradation) - 1`.
The engine's convention, recovered from the output.

In [ ]:
# Revenue decomposition: contracted PPA vs storage arbitrage vs capacity payments.
rev = cf[["stream.energy.ppa.revenue", "stream.energy.storage.margin", "stream.energy.capacity.revenue"]]
rev.groupby(cf.index.year).sum().rename(columns=lambda c: c.split(".")[2]).plot.area(title="Revenue stack by year")

In [ ]:
# Coverage: CFADS against debt service, annually, over the debt's life.
annual = cf[["domain.energy.cfads", "domain.energy.debt_service_periodic"]].groupby(cf.index.year).sum()
live = annual[annual["domain.energy.debt_service_periodic"] > 0]
(live["domain.energy.cfads"] / live["domain.energy.debt_service_periodic"]).plot(title="Annual DSCR (CFADS / debt service)")

In [ ]:
# Equity payback, from the cumulative net line and the engine's own metric.
cum = cf["model.net_cash_flow"].cumsum()
print("first cumulative-positive month:", cum[cum > 0].index.min(),
      "| model.payback_years:", results.metrics()["model.payback_years"])
cum.plot(title="Cumulative net cash flow")